In [9]:
# Model1: XGBoost model to predict Electrical Conductance (EC)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import r2_score

import optuna  # pip install optuna
# XGBoost (install first if needed: `pip install xgboost`)
import xgboost as xgb

In [10]:
# Load engineered training features and join with EC target

features_path = "../New Datasets/Combined/combined_training_engineered.csv"
water_quality_path = "../Provided Datasets/water_quality_training_dataset.csv"

combined = pd.read_csv(features_path)
water_quality = pd.read_csv(water_quality_path)

# Standardize join keys to match `combined`
water_quality_std = water_quality.rename(
    columns={
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Sample Date": "sample_date",
    }
)

# Keep only join keys + EC target
ec_target = water_quality_std[["latitude", "longitude", "sample_date", "Electrical Conductance"]]

# Inner join to align features with EC labels
full = combined.merge(ec_target, on=["latitude", "longitude", "sample_date"], how="inner")

print("Features shape (combined):", combined.shape)
print("Water quality shape:", water_quality.shape)
print("Joined training shape:", full.shape)
full.head()

Features shape (combined): (9319, 82)
Water quality shape: (9319, 6)
Joined training shape: (9319, 83)


,latitude,longitude,sample_date,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,gsw_change,gsw_extent,...,hri,water_perm,water_instab,recurrence_ratio,seasonal_water,esa_change_intensity,wb_x_impervious,eci_x_impervious,gsw_occ_x_impervious,Electrical Conductance
0,-34.405833,19.600556,01-10-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,4703.580299,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1008.0
1,-34.405833,19.600556,02-08-2011,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1984.043203,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1237.0
2,-34.405833,19.600556,02-12-2015,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,6827.433216,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1053.0
3,-34.405833,19.600556,03-07-2013,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,1466.619596,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,1167.0
4,-34.405833,19.600556,03-09-2014,0.0,0.0,0.0,-1.0,-1.0,253.0,0.0,...,2919.631339,0.0,0.0,0.0,0,0.0,-0.0,0.0,0.0,877.0


In [11]:
# Build feature matrix X and target y for EC, then reduce multicollinearity

# Columns to exclude from features
exclude_cols = {
    "Electrical Conductance",            # target
    "Total Alkalinity",                 # do not include
    "Dissolved Reactive Phosphorus",    # do not include
    "latitude", "longitude",           # do not include
    "sample_date",                      # string key, not a feature
}

base_feature_cols = [c for c in full.columns if c not in exclude_cols]
X_full = full[base_feature_cols].copy()
y = full["Electrical Conductance"]

print("Initial number of features:", len(base_feature_cols))

# 1) Remove multicollinearity: drop one of each highly correlated pair
corr_matrix = X_full.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_threshold = 0.95
cols_to_drop_mc = [col for col in upper.columns if any(upper[col] > high_corr_threshold)]

X_mc = X_full.drop(columns=cols_to_drop_mc)
feature_cols_mc = list(X_mc.columns)

print(f"Dropped {len(cols_to_drop_mc)} highly correlated features (>|{high_corr_threshold}|)")
print("Remaining features after multicollinearity reduction:", len(feature_cols_mc))

# For downstream cells we will further reduce by feature importance, then set X and feature_cols there.
# For now, expose X_mc and feature_cols_mc
X_reduced_mc = X_mc.copy()
feature_cols_reduced_mc = feature_cols_mc

print("Example remaining feature columns:", feature_cols_reduced_mc[:10])

Initial number of features: 79
Dropped 24 highly correlated features (>|0.95|)
Remaining features after multicollinearity reduction: 55
Example remaining feature columns: ['gaia_changed_ever_frac', 'gaia_recent_change_5y_frac', 'gsw_change', 'gsw_occurrence', 'gsw_transitions', 'nir', 'green', 'swir16', 'NDMI', 'MNDWI']


In [12]:
# Feature selection via XGBoost feature importance (on multicollinearity-reduced set)

# Use a reasonably strong but not overfitted model to rank features
fs_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

fs_model.fit(X_reduced_mc, y)

importances = fs_model.feature_importances_
fi_fs = pd.DataFrame({"feature": feature_cols_reduced_mc, "importance": importances})
fi_fs = fi_fs.sort_values("importance", ascending=False).reset_index(drop=True)
fi_fs["cum_importance"] = fi_fs["importance"].cumsum()

# Keep features that explain up to 90% of total importance, but ensure at least 20 features
importance_cutoff = 0.90
min_features = 20
selected = fi_fs[fi_fs["cum_importance"] <= importance_cutoff]["feature"].tolist()
if len(selected) < min_features:
    selected = fi_fs.head(min_features)["feature"].tolist()

X = X_reduced_mc[selected].copy()
feature_cols = selected

print("Total features after multicollinearity reduction:", len(feature_cols_reduced_mc))
print("Selected features after importance-based selection:", len(feature_cols))
print("Top selected features:", feature_cols[:15])

Total features after multicollinearity reduction: 55
Selected features after importance-based selection: 26
Top selected features: ['esa_flooded_frac_1km', 'esa_sparse_veg_frac_1km', 'esa_forest_frac_1km', 'esa_grass_frac_1km', 'water_perm', 'recurrence_ratio', 'gsw_recurrence_mean_1km', 'esa_other_frac_1km', 'esa_lccs_class', 'esa_water_frac_1km', 'gsw_occurrence_mean_1km', 'soil', 'gsw_transitions', 'gaia_changed_ever_frac_1km', 'esa_change_count']


In [13]:
# Train XGBoost model and report R² + feature importances

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

# R² scores
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Train R²: {r2_train:.3f}")
print(f"Test  R²: {r2_test:.3f}")

# Feature importances
importances = model.feature_importances_
fi = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi = fi.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC:")
print(fi.head(20).to_string(index=False))

fi.head(20)

Train R²: 0.941
Test  R²: 0.855

Top 20 most important features for predicting EC:
                   feature  importance
        esa_water_frac_1km    0.080212
   gsw_recurrence_mean_1km    0.064581
            esa_lccs_class    0.060999
   gsw_occurrence_mean_1km    0.060504
      esa_flooded_frac_1km    0.057803
                water_perm    0.054374
        esa_grass_frac_1km    0.054277
          recurrence_ratio    0.053385
       esa_forest_frac_1km    0.052938
        esa_other_frac_1km    0.052868
          esa_change_count    0.045377
     esa_cropland_frac_1km    0.044679
        esa_shrub_frac_1km    0.042649
   esa_sparse_veg_frac_1km    0.038886
                      soil    0.032614
        esa_urban_frac_1km    0.032605
           gsw_transitions    0.029880
    gaia_changed_ever_frac    0.026151
                gsw_change    0.025526
gaia_changed_ever_frac_1km    0.024373


,feature,importance
9,esa_water_frac_1km,0.080212
6,gsw_recurrence_mean_1km,0.064581
8,esa_lccs_class,0.060999
10,gsw_occurrence_mean_1km,0.060504
0,esa_flooded_frac_1km,0.057803
4,water_perm,0.054374
3,esa_grass_frac_1km,0.054277
5,recurrence_ratio,0.053385
2,esa_forest_frac_1km,0.052938
7,esa_other_frac_1km,0.052868


In [14]:
# Stratified K-Fold + Optuna hyperparameter tuning for EC model

# Bin the continuous target into quantiles for stratification
n_bins = 10
# qcut can have duplicate bin edges; drop duplicates
y_strat = pd.qcut(y, q=n_bins, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 0.8),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.8),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),
        "gamma": trial.suggest_float("gamma", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    cv_scores = []
    for train_idx, valid_idx in skf.split(X, y_strat):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        y_val_pred = model.predict(X_val)
        cv_scores.append(r2_score(y_val, y_val_pred))

    return float(np.mean(cv_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV R²:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-02-24 17:36:41,498] A new study created in memory with name: no-name-843eb4b8-cf56-4c3f-8072-8e9f53575f09
Best trial: 0. Best value: 0.707191:   2%|▎         | 1/40 [00:02<01:51,  2.87s/it]

[I 2026-02-24 17:36:44,369] Trial 0 finished with value: 0.7071906599028042 and parameters: {'n_estimators': 224, 'max_depth': 4, 'learning_rate': 0.018958661309646543, 'subsample': 0.795379055898638, 'colsample_bytree': 0.5655743440464263, 'min_child_weight': 16.495927307543862, 'gamma': 1.9563627600334774, 'reg_alpha': 0.4536588256416999, 'reg_lambda': 3.1562572831889786}. Best is trial 0 with value: 0.7071906599028042.


Best trial: 1. Best value: 0.778052:   5%|▌         | 2/40 [00:05<01:47,  2.83s/it]

[I 2026-02-24 17:36:47,174] Trial 1 finished with value: 0.7780524169626304 and parameters: {'n_estimators': 223, 'max_depth': 4, 'learning_rate': 0.03612644006279968, 'subsample': 0.6740314847809434, 'colsample_bytree': 0.5711654906433691, 'min_child_weight': 6.550384254076905, 'gamma': 3.9133879882287705, 'reg_alpha': 0.6886057268070482, 'reg_lambda': 4.317127898502302}. Best is trial 1 with value: 0.7780524169626304.


Best trial: 2. Best value: 0.837082:   8%|▊         | 3/40 [00:10<02:22,  3.85s/it]

...

[I 2026-02-24 17:39:35,462] Trial 37 finished with value: 0.8457763875692402 and parameters: {'n_estimators': 377, 'max_depth': 5, 'learning_rate': 0.08545202367068216, 'subsample': 0.6984144340622281, 'colsample_bytree': 0.692328826606052, 'min_child_weight': 5.603908871588197, 'gamma': 0.7418579566597399, 'reg_alpha': 0.667458519197389, 'reg_lambda': 2.024276829735383}. Best is trial 28 with value: 0.8467457822808854.


Best trial: 28. Best value: 0.846746:  98%|█████████▊| 39/40 [02:58<00:05,  5.07s/it]

[I 2026-02-24 17:39:40,199] Trial 38 finished with value: 0.8304833091066275 and parameters: {'n_estimators': 375, 'max_depth': 4, 'learning_rate': 0.05948736966777268, 'subsample': 0.700523290199027, 'colsample_bytree': 0.75668857054051, 'min_child_weight': 11.127608668689774, 'gamma': 0.5581576015632619, 'reg_alpha': 0.7121563059574163, 'reg_lambda': 2.1785338875622475}. Best is trial 28 with value: 0.8467457822808854.


Best trial: 28. Best value: 0.846746: 100%|██████████| 40/40 [03:03<00:00,  4.58s/it]

[I 2026-02-24 17:39:44,684] Trial 39 finished with value: 0.7966377666182813 and parameters: {'n_estimators': 349, 'max_depth': 4, 'learning_rate': 0.02879765499416988, 'subsample': 0.727110223037551, 'colsample_bytree': 0.6968362884328384, 'min_child_weight': 17.69639927763636, 'gamma': 1.1234381238365312, 'reg_alpha': 0.804271446985044, 'reg_lambda': 2.443888029663171}. Best is trial 28 with value: 0.8467457822808854.
Best CV R²: 0.8467457822808854
Best params:
  n_estimators: 357
  max_depth: 5
  learning_rate: 0.08249122310654389
  subsample: 0.7627569037438534
  colsample_bytree: 0.6371781856989825
  min_child_weight: 5.064480501276122
  gamma: 2.224033649170142
  reg_alpha: 0.1197130970552909
  reg_lambda: 1.0323906587708742


In [15]:
# Train final model with best hyperparameters and report R² + feature importances

best_params = study.best_params.copy()
best_params.update({"random_state": 42, "n_jobs": -1})

final_model = xgb.XGBRegressor(**best_params)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Final model (with tuned params) Train R²: {r2_train:.3f}")
print(f"Final model (with tuned params) Test  R²: {r2_test:.3f}")

# Feature importances from tuned model
importances = final_model.feature_importances_
fi_tuned = pd.DataFrame({"feature": feature_cols, "importance": importances})
fi_tuned = fi_tuned.sort_values("importance", ascending=False)

print("\nTop 20 most important features for predicting EC (tuned model):")
print(fi_tuned.head(40).to_string(index=False))

fi_tuned.head(40)

Final model (with tuned params) Train R²: 0.917
Final model (with tuned params) Test  R²: 0.853

Top 20 most important features for predicting EC (tuned model):
                   feature  importance
        esa_water_frac_1km    0.086473
      esa_flooded_frac_1km    0.074073
                water_perm    0.062950
            esa_lccs_class    0.056473
          recurrence_ratio    0.055568
        esa_other_frac_1km    0.053473
       esa_forest_frac_1km    0.051392
   gsw_occurrence_mean_1km    0.050728
        esa_urban_frac_1km    0.050115
        esa_shrub_frac_1km    0.042791
   gsw_recurrence_mean_1km    0.042771
                      soil    0.042761
        esa_grass_frac_1km    0.041515
   esa_sparse_veg_frac_1km    0.040188
          esa_change_count    0.039968
    gaia_changed_ever_frac    0.037143
     esa_cropland_frac_1km    0.032911
gaia_changed_ever_frac_1km    0.028672
      gsw_occ_x_impervious    0.024569
           gsw_transitions    0.022804
                gsw_

,feature,importance
9,esa_water_frac_1km,0.086473
0,esa_flooded_frac_1km,0.074073
4,water_perm,0.062950
8,esa_lccs_class,0.056473
5,recurrence_ratio,0.055568
7,esa_other_frac_1km,0.053473
2,esa_forest_frac_1km,0.051392
10,gsw_occurrence_mean_1km,0.050728
16,esa_urban_frac_1km,0.050115
15,esa_shrub_frac_1km,0.042791


In [16]:
# Create submission1.csv: predictions for EC on the 202 validation rows

# Load submission template (IDs and column order)
sub_template = pd.read_csv("../submission_template.csv")

# Load engineered validation features
val_features = pd.read_csv("../New Datasets/Combined/combined_validation_engineered.csv")

# Standardize keys in template to match engineered features
sub_std = sub_template.rename(columns={
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Sample Date": "sample_date",
})

# Join validation features with template IDs
val_full = val_features.merge(
    sub_std[["latitude", "longitude", "sample_date"]],
    on=["latitude", "longitude", "sample_date"],
    how="inner",
)

print("Validation engineered shape:", val_features.shape)
print("Validation rows with matched features:", val_full.shape[0])

# Build X for validation using the same selected feature set as training
X_val = val_full[feature_cols].copy()

# Predict EC for validation rows
ec_pred = final_model.predict(X_val)

# Build prediction frame with original column names
pred_df = val_full[["latitude", "longitude", "sample_date"]].copy()
pred_df["Electrical Conductance"] = ec_pred
pred_df = pred_df.rename(columns={
    "latitude": "Latitude",
    "longitude": "Longitude",
    "sample_date": "Sample Date",
})

# Start from template and overwrite EC with predictions
submission1 = sub_template.copy()
submission1 = submission1.drop(columns=["Electrical Conductance"]).merge(
    pred_df,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left",
)

out_path = "../submission1.csv"
submission1.to_csv(out_path, index=False)
print("Saved:", out_path)
submission1.head()

Validation engineered shape: (200, 82)
Validation rows with matched features: 200
Saved: ../submission1.csv


,Latitude,Longitude,Sample Date,Total Alkalinity,Dissolved Reactive Phosphorus,Electrical Conductance
0,-32.043333,27.822778,01-09-2014,NaN,NaN,469.420074
1,-33.329167,26.077500,16-09-2015,NaN,NaN,161.709778
2,-32.991639,27.640028,07-05-2015,NaN,NaN,221.352066
3,-34.096389,24.439167,07-02-2012,NaN,NaN,343.663879
4,-32.000556,28.581667,01-10-2014,NaN,NaN,530.699402
